# Acne Detection with YOLOv5

Training YOLOv5s on ACNE04 for multi-class acne lesion detection.
Run this notebook on a **Colab GPU runtime (T4 or better)**.

## Contents
1. Environment setup
2. Data download and config generation  
3. Training (50 epochs, YOLOv5s pretrained on COCO)
4. Detection visualizations on test set
5. Download results
6. Visualize results

## 1) Environment Setup
Clones the repo, installs dependencies, and clones YOLOv5.
**Restart the runtime after this cell completes before continuing.**

In [1]:
!git clone https://github.com/kaizar-rang/yanglab-acne.git
%cd yanglab-acne
!git clone https://github.com/ultralytics/yolov5
%pip install roboflow -q
%pip install -r requirements.txt -q
%pip install -r yolov5/requirements.txt -q
print("\nDone. Restart the runtime now (Runtime -> Restart session), then continue from Cell 2.")

Cloning into 'yanglab-acne'...
remote: Enumerating objects: 3546, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 3546 (delta 26), reused 49 (delta 18), pack-reused 3485 (from 2)
Receiving objects: 100% (3546/3546), 125.32 MiB | 36.58 MiB/s, done.
Resolving deltas: 100% (55/55), done.
/content/yanglab-acne
Cloning into 'yolov5'...
remote: Enumerating objects: 17972, done.
remote: Counting objects: 100% (107/107), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 17972 (delta 81), reused 30 (delta 30), pack-reused 17865 (from 3)
Receiving objects: 100% (17972/17972), 17.09 MiB | 24.79 MiB/s, done.
Resolving deltas: 100% (12226/12226), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.2/249.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 18.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━

## 2) Credentials and Data Download
Enter your Roboflow API key when prompted. Downloads ACNE04 in YOLOv5 format and generates the training config.

In [2]:
import os, sys, yaml, getpass
from pathlib import Path

%cd /content/yanglab-acne
os.environ["WANDB_DISABLED"] = "true"

roboflow_key = getpass.getpass("Enter ROBOFLOW_API_KEY: ")
os.environ["ROBOFLOW_API_KEY"] = roboflow_key

from roboflow import Roboflow
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
version = rf.workspace("acne-vulgaris-detection").project("acne04-detection").version(1)
version.download("yolov5", location="data/acne04", overwrite=True)

repo_root = Path.cwd()
config = {
    "train": str(repo_root / "data/acne04/train/images"),
    "val": str(repo_root / "data/acne04/valid/images"),
    "test": str(repo_root / "data/acne04/test/images"),
    "nc": 4,
    "names": ["nodules and cysts", "papules", "pustules", "whitehead and blackhead"]
}
os.makedirs("configs", exist_ok=True)
with open("configs/acne04.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"Setup complete. Working directory: {os.getcwd()}")

/content/yanglab-acne
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to data/acne04 in yolov5pytorch:: 100%|██████████| 6808/6808 [00:00<00:00, 7688.67it/s]

Setup complete. Working directory: /content/yanglab-acne


## Cell 3: Train YOLOv5s
Trains for 300 epochs with default Ultralytics hyperparameters and COCO pretrained weights.
Expected time: ~45 minutes on T4 GPU.

In [ ]:
!python yolov5/train.py \
    --img 640 \
    --batch 16 \
    --epochs 300 \
    --data configs/acne04.yaml \
    --weights yolov5s.pt \
    --project outputs/checkpoints \
    --name yolov5_acne \
    --exist-ok

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
wandb: WARNING ⚠️ wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: (30 second timeout) 
wandb: WARNING W&B disabled due to login timeout.
wandb: ERROR Error while calling W&B API: api key too short (<Response [401]>)
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
train: weights=yolov5s.pt, cfg=, data=configs/acne04.yaml, hyp=yolov5/data/hyps/hyp.scratch-low.yaml, epochs=50, batch_size=16, imgsz=640, rect=False, resume=False,

In [4]:
!python yolov5/detect.py \
    --weights outputs/checkpoints/yolov5_acne/weights/best.pt \
    --source data/acne04/test/images \
    --data configs/acne04.yaml \
    --conf 0.25 \
    --project outputs/predictions \
    --name yolov5_detections \
    --exist-ok

detect: weights=['outputs/checkpoints/yolov5_acne/weights/best.pt'], source=data/acne04/test/images, data=configs/acne04.yaml, imgsz=[640, 640], conf_thres=0.25, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=False, save_format=0, save_csv=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=outputs/predictions, name=yolov5_detections, exist_ok=True, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-484-g70b964b6 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
Model summary: 157 layers, 7020913 parameters, 0 gradients, 15.8 GFLOPs
image 1/142 /content/yanglab-acne/data/acne04/test/images/levle0_108_jpg.rf.ec4ba68284807cdfb5b46b422cc80ef8.jpg: 640x640 1 pustules, 11.5ms
image 2/142 /content/yanglab-acne/data/acne04/test/images/levle0_10_jpg.rf.243b71b8f6818e3e5c811f4857567bc3.jpg: 640x640 9 papul

## Cell 4: Detection Visualizations
Runs inference on the test set and saves annotated images with bounding boxes to outputs/predictions/.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
import pandas as pd

# Show training curves
img = mpimg.imread('outputs/checkpoints/yolov5_acne/results.png')
plt.figure(figsize=(16, 8))
plt.imshow(img)
plt.axis('off')
plt.title('YOLOv5 Training Results')
plt.show()

# Show PR curve
img = mpimg.imread('outputs/checkpoints/yolov5_acne/PR_curve.png')
plt.figure(figsize=(8, 6))
plt.imshow(img)
plt.axis('off')
plt.title('Precision-Recall Curve')
plt.show()

# Show confusion matrix
img = mpimg.imread('outputs/checkpoints/yolov5_acne/confusion_matrix.png')
plt.figure(figsize=(8, 8))
plt.imshow(img)
plt.axis('off')
plt.title('Confusion Matrix')
plt.show()

# Print final metrics
results = pd.read_csv('outputs/checkpoints/yolov5_acne/results.csv')
results.columns = results.columns.str.strip()
print(results.tail(1).to_string())

## Cell 5: Download Results
**Run immediately after training finishes. Do not close the session first.**
Downloads weights, metrics, and detection images to your local machine.

In [5]:
from google.colab import files

!zip -r yolov5_results.zip outputs/checkpoints/yolov5_acne/
files.download('yolov5_results.zip')

!zip -r yolov5_detections.zip outputs/predictions/yolov5_detections/
files.download('yolov5_detections.zip')

  adding: outputs/checkpoints/yolov5_acne/ (stored 0%)
  adding: outputs/checkpoints/yolov5_acne/val_batch0_labels.jpg (deflated 7%)
  adding: outputs/checkpoints/yolov5_acne/confusion_matrix.png (deflated 22%)
  adding: outputs/checkpoints/yolov5_acne/labels.jpg (deflated 27%)
  adding: outputs/checkpoints/yolov5_acne/val_batch1_labels.jpg (deflated 8%)
  adding: outputs/checkpoints/yolov5_acne/labels_correlogram.jpg (deflated 27%)
  adding: outputs/checkpoints/yolov5_acne/events.out.tfevents.1780689963.00bef18cd693.3042.0 (deflated 27%)
  adding: outputs/checkpoints/yolov5_acne/val_batch2_pred.jpg (deflated 8%)
  adding: outputs/checkpoints/yolov5_acne/opt.yaml (deflated 50%)
  adding: outputs/checkpoints/yolov5_acne/R_curve.png (deflated 11%)
  adding: outputs/checkpoints/yolov5_acne/train_batch1.jpg (deflated 3%)
  adding: outputs/checkpoints/yolov5_acne/val_batch1_pred.jpg (deflated 8%)
  adding: outputs/checkpoints/yolov5_acne/val_batch2_labels.jpg (deflated 8%)
  adding: outputs

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  adding: outputs/predictions/yolov5_detections/ (stored 0%)
  adding: outputs/predictions/yolov5_detections/levle1_144_jpg.rf.ed4b32828ea4ec3bfdb09416f4810c96.jpg (deflated 4%)
  adding: outputs/predictions/yolov5_detections/levle1_69_jpg.rf.98ec93590c2461045738c8c78f39bbba.jpg (deflated 5%)
  adding: outputs/predictions/yolov5_detections/levle1_299_jpg.rf.c0422efac544e440218902e646ddf20e.jpg (deflated 3%)
  adding: outputs/predictions/yolov5_detections/levle0_90_jpg.rf.608b9b085ac1549084e84611f73cfb2f.jpg (deflated 3%)
  adding: outputs/predictions/yolov5_detections/levle1_531_jpg.rf.7fb9de07da378e6b390d3891772a1b89.jpg (deflated 2%)
  adding: outputs/predictions/yolov5_detections/levle0_483_jpg.rf.a1fa8db02cf3419e66c6a84c5ed51e65.jpg (deflated 4%)
  adding: outputs/predictions/yolov5_detections/levle0_424_jpg.rf.b3603c0f0b08a7ac14fbf14d7289b34b.jpg (deflated 4%)
  adding: outputs/predictions/yolov5_detections/levle0_239_jpg.rf.86bac6bf9786cec0c4435179494db0ae.jpg (deflated 4%)
  add

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>